# Chương 4 – RNN đọc chuỗi thời gian

## Cách chạy và phạm vi

    Chọn kernel `.venv` (Python 3.12). Lưu notebook bằng Ctrl+S rồi **Run All**.
    Notebook đọc CSV, dùng toàn bộ train trên 100.000 mẫu của từng bộ, ba mô hình và ba cách cài đặt.
    Các ô bên dưới chứa mã tải/đọc dữ liệu, lớp NumPy tự viết, Keras, PyTorch và vòng học thực sự.
    File `.py` chỉ là nguồn ban đầu để sinh notebook; bạn có thể đọc và chỉnh mã trực tiếp tại đây.

    Chưa có kết quả huấn luyện quy mô lớn. Những ô không có output là chưa chạy, không phải lỗi.
    Không dùng số liệu từ thử nghiệm nhỏ `results/tieuluan/course/` để thay thế.
    Thời gian phụ thuộc CPU, dữ liệu và early stopping; không hứa hẹn cố định 4–5 giờ.
    Sau mỗi epoch lưu cả trọng số, Adam và trạng thái ngẫu nhiên; chạy lại để tiếp tục.
    Checkpoint chỉ đọc tệp cục bộ do bạn tạo, không nhận tệp pickle từ bên ngoài.

In [ ]:
import os, sys, json, hashlib, time
from pathlib import Path
for name in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS'):
    os.environ.setdefault(name,'2')
os.environ.setdefault('KERAS_BACKEND','tensorflow')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL','3')
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/tieuluan/course').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
DATASETS=['jena_10min', 'household_power']
MODEL_KINDS=['rnn', 'lstm', 'gru']
FRAMEWORKS=['scratch','pytorch','keras']
RUN_ID='large_csv_v1'
CONFIG={'seed':11,'epochs':30,'batch_size':128,'patience':5,'learning_rate':0.001}
NOTEBOOK_PATH=ROOT/'notebooks/tieuluan/04_rnn.ipynb'
# Lưu Ctrl+S trước khi chạy để mã được ghi nhận đúng vào dấu vết thực nghiệm.
saved_notebook=json.loads(NOTEBOOK_PATH.read_text(encoding='utf8'))
CODE_HASH=hashlib.sha256('\n'.join(''.join(c['source']) for c in saved_notebook['cells'] if c['cell_type']=='code' and any(t in c.get('metadata',{}).get('tags',[]) for t in ('definition','train'))).encode()).hexdigest()
print('Mã thực nghiệm:',CODE_HASH[:16], '| Thư mục:',RUN_ID)

## 1. Tải và đọc dữ liệu CSV

    Nguồn gốc và checksum được lưu trong manifest. CSV là nguồn dữ liệu notebook đọc;
    cache `.npy` chỉ giúp đọc theo mini-batch mà không tốn hàng GB RAM.
    Ảnh: 784 cột `pixel_*` nguyên 0–255. Bảng/chuỗi: đặc trưng đã chuẩn hóa bằng train.
    `sample_id`, `label`, `previous_direction` không được đưa vào đặc trưng.
    Với chuỗi, các cửa sổ trong một tập chồng lấn và không độc lập; ba tập không dùng chung quan sát.

In [ ]:
"""Dữ liệu quy mô lớn. Các hàm này được chép nguyên mã vào notebook.

Không nhân bản hoặc tăng cường ảnh để đạt ngưỡng 100.000. Không tạo kết quả giả.
Tệp .npy đọc bằng memory mapping, ảnh chỉ đổi sang float32 theo mini-batch.
"""

import gzip

import hashlib

import json

from pathlib import Path

import zipfile

import numpy as np

import pandas as pd

import requests

from sklearn.model_selection import train_test_split

SOURCES={
 'covertype':('Thảm phủ rừng Covertype','tabular','https://archive.ics.uci.edu/dataset/31/covertype'),
 'miniboone':('Nhận dạng hạt MiniBooNE','tabular','https://archive.ics.uci.edu/dataset/199/miniboone+particle+identification'),
 'emnist_digits':('Chữ số EMNIST Digits','image','https://www.nist.gov/itl/products-and-services/emnist-dataset'),
 'k49':('Chữ viết Nhật Kuzushiji-49','image','https://github.com/rois-codh/kmnist'),
 'jena_10min':('Nhiệt độ Jena, chu kỳ 10 phút','sequence','https://weather.bgc-jena.mpg.de/weather_data.html'),
 'household_power':('Điện năng hộ gia đình, trung bình 10 phút','sequence','https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption')}

def enforce_size(n):
    if n<=100_000:
        raise ValueError(f'Tập train phải trên 100.000 mẫu thực, hiện có {n:,}. Không tự nhân bản mẫu.')

def batch_input(x):
    return x.astype('float32')/255 if x.dtype==np.uint8 else np.asarray(x,dtype='float32')

def file_hash(path):
    digest=hashlib.sha256()
    with open(path,'rb') as handle:
        for chunk in iter(lambda:handle.read(1024*1024),b''):
            digest.update(chunk)
    return digest.hexdigest()

def fetch_file(root,url,name):
    folder=Path(root)/'data/tieuluan/course_large/raw'
    folder.mkdir(parents=True,exist_ok=True)
    path=folder/name
    if not path.exists():
        partial=path.with_suffix(path.suffix+'.part')
        print('Đang tải:',name,flush=True)
        with requests.get(url,stream=True,timeout=(15,90),headers={'User-Agent':'Mozilla/5.0'}) as response:
            response.raise_for_status()
            size=0
            with partial.open('wb') as handle:
                for chunk in response.iter_content(1024*1024):
                    handle.write(chunk); size+=len(chunk)
            expected=response.headers.get('Content-Length')
            if expected and not response.headers.get('Content-Encoding') and size!=int(expected):
                raise ValueError('Tệp tải chưa đủ; chạy lại ô tải dữ liệu.')
        partial.replace(path)
    return path,{'url':url,'sha256':file_hash(path),'bytes':path.stat().st_size}

def tabular_arrays(x,y,seed=11):
    ids=np.arange(len(y))
    train,other=train_test_split(ids,test_size=.2,stratify=y,random_state=seed)
    val,test=train_test_split(other,test_size=.5,stratify=y[other],random_state=seed)
    mean=x[train].mean(0,dtype=np.float64)
    scale=x[train].std(0,dtype=np.float64); scale[scale==0]=1
    arrays={'mean':mean,'scale':scale}
    for s,idx in [('train',train),('val',val),('test',test)]:
        arrays.update({f'{s}_x':((x[idx]-mean)/scale).astype('float32'),f'{s}_y':y[idx],f'{s}_ids':idx})
    arrays['sample_raw']=x[test[:10]]
    return arrays

def temporal_arrays(values,window=24):
    """Chia chuỗi gốc trước khi tạo cửa sổ; không dùng chung quan sát giữa ba tập.

    Cửa sổ có NaN bị bỏ, không nối hai phía khoảng trống. Nhãn: bước sau tăng.
    Các cửa sổ trong một tập có thể chồng lấn, không phải mẫu độc lập thống kê.
    """
    values=np.asarray(values,dtype=np.float64)
    a,b=int(len(values)*.7),int(len(values)*.85)
    mean=float(np.nanmean(values[:a])); scale=float(np.nanstd(values[:a]))
    if not np.isfinite(scale) or scale==0:
        raise ValueError('Chuỗi train không có biến thiên hợp lệ.')
    arrays={'mean':np.array([mean]),'scale':np.array([scale])}
    for split,start,end in [('train',0,a),('val',a,b),('test',b,len(values))]:
        windows=np.lib.stride_tricks.sliding_window_view(values[start:end],window+1)
        valid=np.isfinite(windows).all(axis=1)
        sample=windows[valid]
        arrays[f'{split}_x']=((sample[:,:-1]-mean)/scale).astype('float32')[:,:,None]
        arrays[f'{split}_y']=(sample[:,-1]>sample[:,-2]).astype('int64')
        arrays[f'{split}_ids']=np.arange(start+window,end)[valid]
        arrays[f'{split}_persistence']=(sample[:,-2]>sample[:,-3]).astype('int64')
    return arrays,{'window':window,'split_boundaries':[a,b],
        'missing_rows':int(np.isnan(values).sum()),'label':'Bước tiếp theo tăng so với bước cuối cửa sổ',
        'split_policy':'70/15/15 theo thời gian; tạo cửa sổ riêng trong mỗi tập, bỏ mọi cửa sổ chứa thiếu dữ liệu'}

def image_arrays(x,y,xt,yt):
    train,val=train_test_split(np.arange(len(y)),test_size=.1,stratify=y,random_state=11)
    arrays={}
    for s,images,labels,ids in [('train',x[train],y[train],train),('val',x[val],y[val],val),
                               ('test',xt,yt,np.arange(len(yt)))]:
        arrays.update({f'{s}_x':images[:,None].astype('uint8'),f'{s}_y':labels.astype('int64'),f'{s}_ids':ids})
    return arrays

def prepare_dataset(root,key):
    root=Path(root)
    target=root/'data/tieuluan/course_large/prepared'/key
    if (target/'manifest.json').exists():
        meta=json.loads((target/'manifest.json').read_text(encoding='utf8'))
        for name,digest in meta['array_hashes'].items():
            if file_hash(target/name)!=digest:
                raise ValueError('Tài nguyên đã thay đổi: '+name)
        enforce_size(meta['splits']['train']['n'])
        return meta
    sources=[]; extra={}; classes=['Không','Có']
    if key=='covertype':
        path,source=fetch_file(root,'https://archive.ics.uci.edu/static/public/31/covertype.zip','covertype.zip')
        sources.append(source)
        with zipfile.ZipFile(path) as archive:
            with gzip.GzipFile(fileobj=archive.open('covtype.data.gz')) as handle:
                frame=pd.read_csv(handle,header=None)
        x=frame.iloc[:,:-1].to_numpy(dtype='float32')
        y=(frame.iloc[:,-1].to_numpy()==2).astype('int64')
        arrays=tabular_arrays(x,y)
        classes=['Loại rừng khác','Thông lodgepole (lớp gốc 2)']
        extra={'original_rows':len(frame),'task_note':'Bài toán nhị phân lớp 2 so với sáu lớp còn lại; không phải benchmark bảy lớp gốc. Chia ngẫu nhiên chưa chứng minh tổng quát sang vùng địa lý mới.'}
    elif key=='miniboone':
        path,source=fetch_file(root,'https://archive.ics.uci.edu/static/public/199/miniboone+particle+identification.zip','miniboone.zip')
        sources.append(source)
        with zipfile.ZipFile(path) as archive:
            with archive.open('MiniBooNE_PID.txt') as handle:
                signal,background=map(int,handle.readline().split())
                x=np.loadtxt(handle,dtype='float32')
        assert len(x)==signal+background
        y=np.r_[np.ones(signal,dtype='int64'),np.zeros(background,dtype='int64')]
        # Loại bản ghi toàn -999 (không có phép đo hữu ích), công bố số bị loại.
        valid=np.isfinite(x).all(axis=1)&~np.all(x==-999,axis=1)
        arrays=tabular_arrays(x[valid],y[valid])
        classes=['Nền muon','Tín hiệu electron']
        extra={'original_rows':len(x),'removed_rows':int((~valid).sum()),'task_note':'50 phép đo của sự kiện hạt; nhãn suy ra đúng thứ tự và số lượng ở dòng đầu tệp gốc.'}
    elif key=='emnist_digits':
        path,source=fetch_file(root,'https://biometrics.nist.gov/cs_links/EMNIST/gzip.zip','emnist-gzip.zip')
        sources.append(source); values=[]
        with zipfile.ZipFile(path) as archive:
            for split in ('train','test'):
                for field in ('images','labels'):
                    suffix='idx3' if field=='images' else 'idx1'
                    name=f'emnist-digits-{split}-{field}-{suffix}-ubyte.gz'
                    member=next(n for n in archive.namelist() if n.endswith(name))
                    raw=gzip.decompress(archive.read(member))
                    arr=np.frombuffer(raw,dtype='uint8',offset=16 if field=='images' else 8)
                    if field=='images':
                        arr=arr.reshape(-1,28,28).transpose(0,2,1).copy()
                    values.append(arr)
        arrays=image_arrays(*values); classes=[str(i) for i in range(10)]
        extra={'original_rows':len(values[1])+len(values[3]),'official_train':len(values[1]),'official_test':len(values[3]),'task_note':'Giữ tập test chính thức; lấy 10% train chính thức làm validation; chuyển vị ảnh để hiển thị đúng chiều.'}
    elif key=='k49':
        values=[]
        for name in ('k49-train-imgs.npz','k49-train-labels.npz','k49-test-imgs.npz','k49-test-labels.npz'):
            path,source=fetch_file(root,'https://codh.rois.ac.jp/kmnist/dataset/k49/'+name,name)
            sources.append(source)
            with np.load(path) as data:
                values.append(data['arr_0'])
        arrays=image_arrays(*values); classes=[f'Ký tự {i}' for i in range(49)]
        extra={'original_rows':len(values[1])+len(values[3]),'official_train':len(values[1]),'official_test':len(values[3]),'task_note':'49 lớp mất cân bằng; giữ mã nhãn gốc 0–48. Ghi công CODH/NIJL, giấy phép CC BY-SA 4.0. Không phải KMNIST 10 lớp.'}
    elif key in ('jena_10min','household_power'):
        if key=='jena_10min':
            path,source=fetch_file(root,'https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip','jena.zip')
            with zipfile.ZipFile(path) as archive:
                frame=pd.read_csv(archive.open('jena_climate_2009_2016.csv'))
            dates=pd.to_datetime(frame['Date Time'],format='%d.%m.%Y %H:%M:%S')
            series=pd.Series(frame['T (degC)'].to_numpy(),index=dates).sort_index()
            series=series.groupby(level=0).mean().resample('10min').mean()
            series[series<=-999]=np.nan
        else:
            path,source=fetch_file(root,'https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip','household.zip')
            with zipfile.ZipFile(path) as archive:
                frame=pd.read_csv(archive.open('household_power_consumption.txt'),sep=';',na_values=['?'],
                    usecols=['Date','Time','Global_active_power'])
            dates=pd.to_datetime(frame['Date']+' '+frame['Time'],format='%d/%m/%Y %H:%M:%S')
            raw=pd.Series(frame['Global_active_power'].to_numpy(dtype=float),index=dates)
            # Chỉ dùng khoảng có đủ 10 quan trắc; không điền bằng dữ liệu tương lai.
            series=raw.resample('10min').mean()
            series[raw.resample('10min').count()<10]=np.nan
        sources.append(source)
        arrays,extra=temporal_arrays(series.to_numpy())
        arrays['raw_values']=series.to_numpy(); arrays['raw_dates']=series.index.to_numpy(dtype='datetime64[s]')
        extra.update({'original_rows':len(frame),'regular_rows':len(series),'interval_minutes':10})
        classes=['Không tăng','Tăng']
    else:
        raise ValueError('Bộ dữ liệu không được hỗ trợ: '+key)
    enforce_size(len(arrays['train_y']))
    target.mkdir(parents=True,exist_ok=True)
    hashes={}
    for name,value in arrays.items():
        np.save(target/f'{name}.npy',value,allow_pickle=False)
        hashes[name+'.npy']=file_hash(target/f'{name}.npy')
    name,group,page=SOURCES[key]
    meta={'key':key,'name':name,'group':group,'source_page':page,'sources':sources,
        'classes':classes,'input_shape':list(arrays['train_x'].shape[1:]),'array_hashes':hashes,**extra,
        'splits':{s:{'n':len(arrays[f'{s}_y']),'class_counts':np.bincount(arrays[f'{s}_y'],minlength=len(classes)).tolist()} for s in ('train','val','test')}}
    (target/'manifest.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf8')
    return meta

def load_dataset(root,key):
    folder=Path(root)/'data/tieuluan/course_large/prepared'/key
    meta=json.loads((folder/'manifest.json').read_text(encoding='utf8'))
    enforce_size(meta['splits']['train']['n'])
    arrays={p.stem:np.load(p,mmap_mode='r',allow_pickle=False) for p in folder.glob('*.npy')}
    return arrays,meta

In [ ]:
"""Xuất CSV và đọc CSV theo khối; bộ nhớ đệm không thay thế nguồn CSV.

Ảnh dùng 784 cột điểm ảnh uint8. Bảng và cửa sổ chuỗi giữ số float32 đã
chuẩn hóa bằng tập train. Thứ tự nhãn/định danh được giữ nguyên.
"""

import json

import hashlib

from pathlib import Path

import numpy as np

import pandas as pd

SPLIT_FILES={'train':'train.csv','val':'validation.csv','test':'test.csv'}

def export_dataset_csv(root,key,overwrite=False):
    base=Path(root)/'data/tieuluan/course_large'
    source=base/'prepared'/key
    target=base/'csv'/key
    meta=json.loads((source/'manifest.json').read_text(encoding='utf8'))
    if (target/'manifest.json').exists() and not overwrite:
        return json.loads((target/'manifest.json').read_text(encoding='utf8'))
    target.mkdir(parents=True,exist_ok=True)
    width=int(np.prod(meta['input_shape']))
    prefix={'image':'pixel','tabular':'feature','sequence':'step'}[meta['group']]
    names=[f'{prefix}_{i:03d}' for i in range(width)]
    hashes={}
    for split,name in SPLIT_FILES.items():
        x=np.load(source/f'{split}_x.npy',mmap_mode='r')
        y=np.load(source/f'{split}_y.npy',mmap_mode='r')
        ids=np.load(source/f'{split}_ids.npy',mmap_mode='r')
        persistence_path=source/f'{split}_persistence.npy'
        persistence=np.load(persistence_path,mmap_mode='r') if persistence_path.exists() else None
        temporary=target/(name+'.part')
        with temporary.open('w',encoding='utf8',newline='') as handle:
            for start in range(0,len(y),5000):
                end=min(len(y),start+5000)
                frame=pd.DataFrame(x[start:end].reshape(end-start,-1),columns=names)
                frame.insert(0,'sample_id',ids[start:end])
                frame['label']=y[start:end]
                if persistence is not None:
                    frame['previous_direction']=persistence[start:end]
                frame.to_csv(handle,index=False,header=start==0,float_format='%.9g')
        temporary.replace(target/name)
        hashes[name]=file_hash(target/name)
        print(key,name,len(y),flush=True)
    if (source/'raw_dates.npy').exists():
        pd.DataFrame({'timestamp':np.load(source/'raw_dates.npy'),
                      'value':np.load(source/'raw_values.npy')}).to_csv(target/'observations.csv',index=False)
    description=('Mỗi ảnh 28×28 được trải theo hàng thành 784 số nguyên 0–255; label là lớp gốc.'
        if meta['group']=='image' else 'Các cột đặc trưng đã chuẩn hóa bằng thống kê chỉ từ train; không chuẩn hóa lại trên toàn bộ CSV.')
    meta.update({'csv_features':names,'csv_hashes':hashes,'csv_description':description,
        'csv_extra_columns':['sample_id','label','previous_direction'],
        'csv_id_note':'ID của test ảnh thuộc tập test chính thức riêng; số ID có thể trùng số ID train nhưng không cùng ảnh.'})
    (target/'manifest.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf8')
    (target/'README.md').write_text('# '+meta['name']+'\n\n'+description+
        '\n\n`sample_id` và `label` không được đưa vào đặc trưng. `previous_direction`, nếu có, chỉ dùng cho đường cơ sở. '+
        'Các cột dùng để học được liệt kê trong manifest.json. CSV dùng dấu chấm thập phân và dấu phẩy ngăn cột theo quy ước Python.\n'+
        '\nNguồn gốc: '+meta.get('source_page','Dữ liệu kiểm thử')+'\n',encoding='utf8')
    return meta

def load_csv_dataset(root,key,require_large=True):
    """Đọc CSV thật; đổi nội dung CSV sẽ đổi hash và làm mới cache NumPy.

    Memory mapping chỉ là bộ đệm để không nạp hàng trăm nghìn ảnh vào RAM.
    Không đọc các đặc trưng từ thư mục prepared/.
    """
    base=Path(root)/'data/tieuluan/course_large'
    folder=base/'csv'/key
    meta=json.loads((folder/'manifest.json').read_text(encoding='utf8'))
    hashes={name:file_hash(folder/name) for name in SPLIT_FILES.values()}
    identity=hashlib.sha256(json.dumps({'files':hashes,'shape':meta['input_shape'],
        'features':meta['csv_features']},sort_keys=True).encode()).hexdigest()[:20]
    cache=base/'csv_cache'/key/identity
    cache.mkdir(parents=True,exist_ok=True)
    marker=cache/'hashes.json'
    current=json.loads(marker.read_text()) if marker.exists() else {}
    names=meta['csv_features']; shape=tuple(meta['input_shape'])
    dtype='uint8' if meta['group']=='image' else 'float32'
    if hashes!=current:
        for split,name in SPLIT_FILES.items():
            n=meta['splits'][split]['n']
            x=np.lib.format.open_memmap(cache/f'{split}_x.npy',mode='w+',dtype=dtype,shape=(n,*shape))
            y=np.lib.format.open_memmap(cache/f'{split}_y.npy',mode='w+',dtype='int64',shape=(n,))
            ids=np.lib.format.open_memmap(cache/f'{split}_ids.npy',mode='w+',dtype='int64',shape=(n,))
            offset=0; persistence=None
            # Float64 trước khi kiểm tra ngăn số ngoài 0–255 bị cuộn vòng uint8.
            for chunk in pd.read_csv(folder/name,chunksize=5000):
                end=offset+len(chunk)
                if end>n:
                    raise ValueError('Số dòng CSV khác manifest: '+name)
                values=chunk[names].to_numpy(dtype='float64')
                labels=chunk['label'].to_numpy(dtype='float64')
                if not np.isfinite(values).all() or not np.isfinite(labels).all():
                    raise ValueError('CSV chứa đặc trưng/nhãn thiếu hoặc vô hạn.')
                if np.any(labels!=np.floor(labels)) or np.any(labels<0) or np.any(labels>=len(meta['classes'])):
                    raise ValueError('Nhãn CSV ngoài danh mục lớp.')
                if dtype=='uint8' and (np.any(values<0) or np.any(values>255) or np.any(values!=np.floor(values))):
                    raise ValueError('Điểm ảnh CSV phải là số nguyên từ 0 đến 255.')
                x[offset:end]=values.reshape(-1,*shape)
                y[offset:end]=labels.astype('int64')
                ids[offset:end]=chunk['sample_id'].to_numpy(dtype='int64')
                if 'previous_direction' in chunk:
                    if persistence is None:
                        persistence=np.lib.format.open_memmap(cache/f'{split}_persistence.npy',mode='w+',dtype='int64',shape=(n,))
                    persistence[offset:end]=chunk['previous_direction'].to_numpy(dtype='int64')
                offset=end
            if offset!=n:
                raise ValueError('Số dòng CSV khác manifest: '+name)
            x.flush(); y.flush(); ids.flush()
            if persistence is not None:
                persistence.flush()
            del x,y,ids,persistence
        marker.write_text(json.dumps(hashes,sort_keys=True),encoding='utf8')
    arrays={p.stem:np.load(p,mmap_mode='r',allow_pickle=False) for p in cache.glob('*.npy')}
    if require_large:
        enforce_size(len(arrays['train_y']))
    meta['csv_hashes']=hashes
    # Signature thực nghiệm phải theo CSV đã đọc, không theo cache cũ.
    meta['array_hashes']=hashes
    return arrays,meta

In [ ]:
for key in DATASETS:
    csv_manifest=ROOT/'data/tieuluan/course_large/csv'/key/'manifest.json'
    if not csv_manifest.exists():
        prepare_dataset(ROOT,key)
        export_dataset_csv(ROOT,key)
    data,meta=load_csv_dataset(ROOT,key)
    print(meta['name'],meta['source_page'])
    display(pd.DataFrame([{'Tập':s,'Số mẫu':m['n'],'Phân bố lớp':str(m['class_counts'])}
        for s,m in meta['splits'].items()]))
    print(meta.get('task_note',meta.get('split_policy','')))
    assert len(data['train_y'])>100000
    display(pd.read_csv(ROOT/'data/tieuluan/course_large/csv'/key/'train.csv',nrows=5))
    fig,ax=plt.subplots(figsize=(9,3))
    ax.bar(np.arange(len(meta['classes'])),np.bincount(data['train_y']))
    ax.set(title=meta['name']+' – phân bố tập học',xlabel='Mã lớp',ylabel='Số mẫu')
    plt.show()
    if meta['group']=='image':
        fig,axes=plt.subplots(2,5,figsize=(10,4))
        for i,ax in enumerate(axes.flat):
            ax.imshow(data['train_x'][i,0],cmap='gray'); ax.set_title('Nhãn '+str(data['train_y'][i])); ax.axis('off')
        plt.show()
    elif meta['group']=='sequence':
        plt.plot(data['train_x'][0,:,0]); plt.title('Cửa sổ đầu vào đã chuẩn hóa'); plt.show()
    del data

## 2. NumPy tự viết: tham số, lan truyền xuôi và lan truyền ngược

    Các lớp dưới đây có mã trực tiếp, không gọi scikit-learn để huấn luyện bản scratch.
    Conv2d dùng im2col; RNN/LSTM/GRU dùng lan truyền ngược theo thời gian.
    Các lớp không dùng trong chương vẫn được giữ để có cùng nền tảng kiểm chứng.

### Mã nền tảng: core

In [ ]:
"""Khối cơ sở: tham số, lớp (module) và mô hình tuần tự."""

from pathlib import Path

import numpy as np

class Parameter:
    """Một mảng trọng số cùng gradient của nó (``grad`` có cùng kích thước với ``value``)."""

    def __init__(self, value: np.ndarray, name: str = ""):
        self.value = value
        self.grad = np.zeros_like(value)
        self.name = name

    def zero_grad(self) -> None:
        self.grad[...] = 0.0

class Module:
    """Lớp cơ sở. Lớp con cài ``forward`` và ``backward``; ``training`` bật/tắt Dropout, BatchNorm."""

    training: bool = True

    def forward(self, x: np.ndarray) -> np.ndarray:  # pragma: no cover - lớp trừu tượng
        raise NotImplementedError

    def backward(self, grad: np.ndarray) -> np.ndarray:  # pragma: no cover - lớp trừu tượng
        raise NotImplementedError

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return self.forward(x)

    def parameters(self) -> list[Parameter]:
        params = []
        for value in self.__dict__.values():
            if isinstance(value, Parameter):
                params.append(value)
            elif isinstance(value, Module):
                params.extend(value.parameters())
            elif isinstance(value, (list, tuple)):
                params.extend(p for item in value if isinstance(item, Module) for p in item.parameters())
        return params

    def buffers(self) -> dict[str, np.ndarray]:
        """Trạng thái không học bằng gradient (ví dụ thống kê chạy của BatchNorm)."""
        return {}

    def train(self, mode: bool = True) -> "Module":
        self.training = mode
        for value in self.__dict__.values():
            if isinstance(value, Module):
                value.train(mode)
            elif isinstance(value, (list, tuple)):
                for item in value:
                    if isinstance(item, Module):
                        item.train(mode)
        return self

    def eval(self) -> "Module":
        return self.train(False)

    def zero_grad(self) -> None:
        for param in self.parameters():
            param.zero_grad()

    def num_parameters(self) -> int:
        return int(sum(p.value.size for p in self.parameters()))

class Sequential(Module):
    """Xếp chồng các lớp: đầu ra lớp trước là đầu vào lớp sau; lan truyền ngược theo chiều ngược lại."""

    def __init__(self, *layers: Module):
        self.layers = list(layers)

    def forward(self, x: np.ndarray) -> np.ndarray:
        for layer in self.layers:
            x = layer.forward(x)
        return x

    def backward(self, grad: np.ndarray) -> np.ndarray:
        for layer in reversed(self.layers):
            grad = layer.backward(grad)
        return grad

    def state_dict(self) -> dict[str, np.ndarray]:
        state = {}
        for index, layer in enumerate(self.layers):
            for p_index, param in enumerate(layer.parameters()):
                state[f"{index}.{param.name or p_index}"] = param.value.copy()
            for b_name, buf in layer.buffers().items():
                state[f"{index}.{b_name}"] = buf.copy()
        return state

    def load_state_dict(self, state: dict[str, np.ndarray]) -> None:
        for index, layer in enumerate(self.layers):
            for p_index, param in enumerate(layer.parameters()):
                param.value[...] = state[f"{index}.{param.name or p_index}"]
            for b_name, buf in layer.buffers().items():
                buf[...] = state[f"{index}.{b_name}"]

    def save(self, path: str | Path) -> None:
        np.savez_compressed(path, **self.state_dict())

    def load(self, path: str | Path) -> "Sequential":
        with np.load(path) as data:
            self.load_state_dict({key: data[key] for key in data.files})
        return self

def glorot_uniform(rng: np.random.Generator, fan_in: int, fan_out: int, shape, dtype) -> np.ndarray:
    """Khởi tạo Glorot/Xavier đều: U(-a, a), a = sqrt(6 / (fan_in + fan_out)). Mặc định của Keras."""
    limit = np.sqrt(6.0 / (fan_in + fan_out))
    return rng.uniform(-limit, limit, size=shape).astype(dtype)

def orthogonal(rng: np.random.Generator, rows: int, cols: int, dtype) -> np.ndarray:
    """Ma trận trực giao (Saxe et al., 2014) — mặc định của Keras cho trọng số hồi quy."""
    flat = rng.normal(0.0, 1.0, size=(max(rows, cols), min(rows, cols)))
    q, r = np.linalg.qr(flat)
    q = q * np.sign(np.diag(r))
    if rows < cols:
        q = q.T
    return q[:rows, :cols].astype(dtype)

### Mã nền tảng: layers

In [ ]:
"""Các lớp cơ bản: Dense, hàm kích hoạt, Dropout, BatchNorm, Conv2d (im2col), MaxPool2d.

Quy ước kích thước (giống PyTorch): ảnh có dạng [N, C, H, W]; trọng số Dense có dạng
[out, in]; trọng số Conv2d có dạng [F, C, KH, KW]. Gradient được *cộng dồn* vào
``Parameter.grad`` nên cần gọi ``zero_grad()`` trước mỗi bước cập nhật.
"""

import numpy as np

from numpy.lib.stride_tricks import as_strided

def _pair(value) -> tuple[int, int]:
    return (int(value), int(value)) if np.isscalar(value) else (int(value[0]), int(value[1]))

class Dense(Module):
    """Lớp kết nối đầy đủ: y = x·Wᵀ + b."""

    def __init__(self, in_features: int, out_features: int, *, rng=None, dtype=np.float32):
        rng = rng or np.random.default_rng(0)
        self.weight = Parameter(glorot_uniform(rng, in_features, out_features, (out_features, in_features), dtype), "weight")
        self.bias = Parameter(np.zeros(out_features, dtype=dtype), "bias")
        self._x = None

    def forward(self, x):
        self._x = x
        return x @ self.weight.value.T + self.bias.value

    def backward(self, grad):
        self.weight.grad += grad.T @ self._x
        self.bias.grad += grad.sum(axis=0)
        return grad @ self.weight.value

class ReLU(Module):
    def forward(self, x):
        self._mask = x > 0
        return np.where(self._mask, x, 0).astype(x.dtype, copy=False)

    def backward(self, grad):
        return grad * self._mask

class LeakyReLU(Module):
    """f(x) = x nếu x > 0, ngược lại α·x (Jiang, Kelly & Xiu 2023 dùng α = 0,01)."""

    def __init__(self, negative_slope: float = 0.01):
        self.alpha = negative_slope

    def forward(self, x):
        self._mask = x > 0
        return np.where(self._mask, x, self.alpha * x).astype(x.dtype, copy=False)

    def backward(self, grad):
        return grad * np.where(self._mask, 1.0, self.alpha).astype(grad.dtype, copy=False)

class Sigmoid(Module):
    def forward(self, x):
        self._y = 0.5 * (1.0 + np.tanh(0.5 * x))  # dạng ổn định số học của 1/(1+e^-x)
        return self._y

    def backward(self, grad):
        return grad * self._y * (1.0 - self._y)

class Tanh(Module):
    def forward(self, x):
        self._y = np.tanh(x)
        return self._y

    def backward(self, grad):
        return grad * (1.0 - self._y ** 2)

class Dropout(Module):
    """Dropout "đảo ngược": khi huấn luyện giữ lại mỗi nơ-ron với xác suất 1-p rồi chia cho 1-p."""

    def __init__(self, p: float = 0.5, *, rng=None):
        self.p = float(p)
        self.rng = rng or np.random.default_rng(0)
        self._mask = None

    def forward(self, x):
        if not self.training or self.p == 0.0:
            self._mask = None
            return x
        keep = 1.0 - self.p
        self._mask = (self.rng.random(x.shape) < keep).astype(x.dtype) / keep
        return x * self._mask

    def backward(self, grad):
        return grad if self._mask is None else grad * self._mask

class Flatten(Module):
    def forward(self, x):
        self._shape = x.shape
        return x.reshape(x.shape[0], -1)

    def backward(self, grad):
        return grad.reshape(self._shape)

class _BatchNorm(Module):
    """Chuẩn hóa theo lô (Ioffe & Szegedy, 2015). ``momentum`` theo quy ước PyTorch (0,1)."""

    axes: tuple[int, ...] = (0,)

    def __init__(self, num_features: int, *, eps: float = 1e-5, momentum: float = 0.1, dtype=np.float32):
        self.gamma = Parameter(np.ones(num_features, dtype=dtype), "gamma")
        self.beta = Parameter(np.zeros(num_features, dtype=dtype), "beta")
        self.running_mean = np.zeros(num_features, dtype=dtype)
        self.running_var = np.ones(num_features, dtype=dtype)
        self.eps, self.momentum = eps, momentum

    def _shape(self, v, ndim):
        return v.reshape((1, -1) + (1,) * (ndim - 2))

    def forward(self, x):
        if self.training:
            mean = x.mean(axis=self.axes)
            var = x.var(axis=self.axes)
            count = x.size // x.shape[1]
            unbiased = var * count / max(count - 1, 1)
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * unbiased
        else:
            mean, var = self.running_mean, self.running_var
        inv_std = 1.0 / np.sqrt(var + self.eps)
        x_hat = (x - self._shape(mean, x.ndim)) * self._shape(inv_std, x.ndim)
        self._cache = (x_hat, inv_std)
        return self._shape(self.gamma.value, x.ndim) * x_hat + self._shape(self.beta.value, x.ndim)

    def backward(self, grad):
        x_hat, inv_std = self._cache
        self.gamma.grad += (grad * x_hat).sum(axis=self.axes)
        self.beta.grad += grad.sum(axis=self.axes)
        g = grad * self._shape(self.gamma.value, grad.ndim)
        if not self.training:  # chế độ suy luận: phép biến đổi tuyến tính cố định
            return g * self._shape(inv_std, grad.ndim)
        m = grad.size // grad.shape[1]
        sum_g = g.sum(axis=self.axes, keepdims=True)
        sum_gx = (g * x_hat).sum(axis=self.axes, keepdims=True)
        return (self._shape(inv_std, grad.ndim) / m) * (m * g - sum_g - x_hat * sum_gx)

    def buffers(self):
        return {"running_mean": self.running_mean, "running_var": self.running_var}

class BatchNorm1d(_BatchNorm):
    axes = (0,)

class BatchNorm2d(_BatchNorm):
    axes = (0, 2, 3)

class Conv2d(Module):
    """Tích chập 2 chiều cài bằng kỹ thuật im2col.

    Mỗi "cửa sổ" KH×KW của ảnh được duỗi thành một hàng của ma trận ``cols``; khi đó
    phép tích chập trở thành một phép nhân ma trận lớn (cols · Wᵀ) — cách các thư viện
    như Caffe từng dùng. Hỗ trợ stride, padding và dilation giống ``torch.nn.Conv2d``.
    """

    def __init__(self, in_channels, out_channels, kernel_size, *, stride=1, padding=0, dilation=1,
                 rng=None, dtype=np.float32):
        rng = rng or np.random.default_rng(0)
        self.kernel = _pair(kernel_size)
        self.stride, self.padding, self.dilation = _pair(stride), _pair(padding), _pair(dilation)
        kh, kw = self.kernel
        fan_in, fan_out = in_channels * kh * kw, out_channels * kh * kw
        self.weight = Parameter(glorot_uniform(rng, fan_in, fan_out, (out_channels, in_channels, kh, kw), dtype), "weight")
        self.bias = Parameter(np.zeros(out_channels, dtype=dtype), "bias")

    def output_shape(self, h: int, w: int) -> tuple[int, int]:
        (kh, kw), (sh, sw), (ph, pw), (dh, dw) = self.kernel, self.stride, self.padding, self.dilation
        return (h + 2 * ph - dh * (kh - 1) - 1) // sh + 1, (w + 2 * pw - dw * (kw - 1) - 1) // sw + 1

    def forward(self, x):
        n, c, h, w = x.shape
        (kh, kw), (sh, sw), (ph, pw), (dh, dw) = self.kernel, self.stride, self.padding, self.dilation
        xp = np.pad(x, ((0, 0), (0, 0), (ph, ph), (pw, pw))) if (ph or pw) else np.ascontiguousarray(x)
        oh, ow = self.output_shape(h, w)
        s_n, s_c, s_h, s_w = xp.strides
        windows = as_strided(xp, shape=(n, oh, ow, c, kh, kw),
                             strides=(s_n, s_h * sh, s_w * sw, s_c, s_h * dh, s_w * dw), writeable=False)
        cols = windows.reshape(n * oh * ow, c * kh * kw)
        out = cols @ self.weight.value.reshape(self.weight.value.shape[0], -1).T + self.bias.value
        self._cache = (cols, x.shape, xp.shape, oh, ow)
        return np.ascontiguousarray(out.reshape(n, oh, ow, -1).transpose(0, 3, 1, 2))

    def backward(self, grad):
        cols, (n, c, h, w), xp_shape, oh, ow = self._cache
        (kh, kw), (sh, sw), (ph, pw), (dh, dw) = self.kernel, self.stride, self.padding, self.dilation
        f = self.weight.value.shape[0]
        g = grad.transpose(0, 2, 3, 1).reshape(n * oh * ow, f)
        self.weight.grad += (g.T @ cols).reshape(self.weight.value.shape)
        self.bias.grad += g.sum(axis=0)
        dcols = (g @ self.weight.value.reshape(f, -1)).reshape(n, oh, ow, c, kh, kw)
        dxp = np.zeros(xp_shape, dtype=grad.dtype)
        for i in range(kh):  # "col2im": trả gradient của từng vị trí trong nhân về đúng điểm ảnh
            for j in range(kw):
                hs, ws = i * dh, j * dw
                dxp[:, :, hs:hs + sh * (oh - 1) + 1:sh, ws:ws + sw * (ow - 1) + 1:sw] += \
                    dcols[:, :, :, :, i, j].transpose(0, 3, 1, 2)
        return dxp[:, :, ph:ph + h, pw:pw + w]

class MaxPool2d(Module):
    """Gộp cực đại. Khi có nhiều giá trị bằng nhau, chọn phần tử xuất hiện đầu tiên (giống PyTorch)."""

    def __init__(self, kernel_size, stride=None):
        self.kernel = _pair(kernel_size)
        self.stride = _pair(stride) if stride is not None else self.kernel

    def forward(self, x):
        x = np.ascontiguousarray(x)
        n, c, h, w = x.shape
        (kh, kw), (sh, sw) = self.kernel, self.stride
        oh, ow = (h - kh) // sh + 1, (w - kw) // sw + 1
        s_n, s_c, s_h, s_w = x.strides
        windows = as_strided(x, shape=(n, c, oh, ow, kh, kw),
                             strides=(s_n, s_c, s_h * sh, s_w * sw, s_h, s_w), writeable=False)
        flat = windows.reshape(n, c, oh, ow, kh * kw)
        idx = flat.argmax(axis=-1)
        self._cache = (x.shape, idx, oh, ow)
        return np.take_along_axis(flat, idx[..., None], axis=-1)[..., 0]

    def backward(self, grad):
        shape, idx, oh, ow = self._cache
        (kh, kw), (sh, sw) = self.kernel, self.stride
        n, c = shape[:2]
        rows = np.arange(oh)[None, None, :, None] * sh + idx // kw
        cols = np.arange(ow)[None, None, None, :] * sw + idx % kw
        nn_ = np.arange(n)[:, None, None, None]
        cc = np.arange(c)[None, :, None, None]
        dx = np.zeros(shape, dtype=grad.dtype)
        if (sh, sw) == (kh, kw):  # cửa sổ không chồng lấn → mỗi vị trí nhận tối đa một gradient
            dx[nn_, cc, rows, cols] = grad
        else:
            np.add.at(dx, (nn_, cc, rows, cols), grad)
        return dx

### Mã nền tảng: recurrent

In [ ]:
"""Mạng hồi quy: SimpleRNN (Elman), LSTM, GRU với lan truyền ngược theo thời gian (BPTT).

Đầu vào dạng [N, T, D] (batch_first), đầu ra là toàn bộ chuỗi trạng thái ẩn [N, T, H];
dùng ``LastStep`` để lấy trạng thái cuối cho bài toán "nhiều-đến-một" (many-to-one).
Bố cục trọng số trùng PyTorch: weight_ih [G·H, D], weight_hh [G·H, H], hai vector bias,
thứ tự cổng LSTM (i, f, g, o) và GRU (r, z, n). Khởi tạo theo mặc định của Keras:
Glorot đều cho trọng số đầu vào, trực giao cho trọng số hồi quy, bias 0 và bias cổng quên = 1.
"""

import numpy as np

def _sigmoid(x):
    return 0.5 * (1.0 + np.tanh(0.5 * x))

class _Recurrent(Module):
    gates = 1

    def __init__(self, input_size: int, hidden_size: int, *, rng=None, dtype=np.float32, forget_bias: float = 0.0):
        rng = rng or np.random.default_rng(0)
        g, h = self.gates, hidden_size
        self.hidden_size = h
        self.weight_ih = Parameter(glorot_uniform(rng, input_size, g * h, (g * h, input_size), dtype), "weight_ih")
        self.weight_hh = Parameter(orthogonal(rng, g * h, h, dtype), "weight_hh")
        bias = np.zeros(g * h, dtype=dtype)
        if forget_bias:
            bias[h:2 * h] = forget_bias
        self.bias_ih = Parameter(bias, "bias_ih")
        self.bias_hh = Parameter(np.zeros(g * h, dtype=dtype), "bias_hh")

class SimpleRNN(_Recurrent):
    """h_t = tanh(W_ih·x_t + b_ih + W_hh·h_{t-1} + b_hh)."""

    gates = 1

    def forward(self, x):
        n, t_len, _ = x.shape
        h = np.zeros((n, self.hidden_size), dtype=x.dtype)
        xw = x @ self.weight_ih.value.T + self.bias_ih.value
        hs = np.empty((n, t_len + 1, self.hidden_size), dtype=x.dtype)
        hs[:, 0] = h
        for t in range(t_len):
            h = np.tanh(xw[:, t] + h @ self.weight_hh.value.T + self.bias_hh.value)
            hs[:, t + 1] = h
        self._cache = (x, hs)
        return hs[:, 1:]

    def backward(self, grad):
        x, hs = self._cache
        n, t_len, d = x.shape
        dxw = np.empty((n, t_len, self.hidden_size), dtype=grad.dtype)
        dh_next = np.zeros((n, self.hidden_size), dtype=grad.dtype)
        w_hh = self.weight_hh.value
        for t in reversed(range(t_len)):
            da = (grad[:, t] + dh_next) * (1.0 - hs[:, t + 1] ** 2)
            self.weight_hh.grad += da.T @ hs[:, t]
            dxw[:, t] = da
            dh_next = da @ w_hh
        flat = dxw.reshape(-1, self.hidden_size)
        self.weight_ih.grad += flat.T @ x.reshape(-1, d)
        self.bias_ih.grad += flat.sum(axis=0)
        self.bias_hh.grad += flat.sum(axis=0)
        return dxw @ self.weight_ih.value

class LSTM(_Recurrent):
    """LSTM (Hochreiter & Schmidhuber, 1997; cổng quên của Gers et al., 2000).

    i = σ(·), f = σ(·), g = tanh(·), o = σ(·);  c_t = f⊙c_{t-1} + i⊙g;  h_t = o⊙tanh(c_t).
    """

    gates = 4

    def __init__(self, input_size, hidden_size, *, rng=None, dtype=np.float32, forget_bias: float = 1.0):
        super().__init__(input_size, hidden_size, rng=rng, dtype=dtype, forget_bias=forget_bias)

    def forward(self, x):
        n, t_len, _ = x.shape
        hd = self.hidden_size
        xw = x @ self.weight_ih.value.T + self.bias_ih.value
        h = np.zeros((n, hd), dtype=x.dtype)
        c = np.zeros((n, hd), dtype=x.dtype)
        hs = np.empty((n, t_len + 1, hd), dtype=x.dtype); hs[:, 0] = h
        cs = np.empty((n, t_len + 1, hd), dtype=x.dtype); cs[:, 0] = c
        acts = np.empty((n, t_len, 4 * hd), dtype=x.dtype)
        tcs = np.empty((n, t_len, hd), dtype=x.dtype)
        for t in range(t_len):
            z = xw[:, t] + h @ self.weight_hh.value.T + self.bias_hh.value
            i = _sigmoid(z[:, :hd]); f = _sigmoid(z[:, hd:2 * hd])
            g = np.tanh(z[:, 2 * hd:3 * hd]); o = _sigmoid(z[:, 3 * hd:])
            c = f * c + i * g
            tc = np.tanh(c)
            h = o * tc
            acts[:, t] = np.concatenate([i, f, g, o], axis=1)
            tcs[:, t], hs[:, t + 1], cs[:, t + 1] = tc, h, c
        self._cache = (x, hs, cs, acts, tcs)
        return hs[:, 1:]

    def backward(self, grad):
        x, hs, cs, acts, tcs = self._cache
        n, t_len, d = x.shape
        hd = self.hidden_size
        dz_all = np.empty((n, t_len, 4 * hd), dtype=grad.dtype)
        dh_next = np.zeros((n, hd), dtype=grad.dtype)
        dc_next = np.zeros((n, hd), dtype=grad.dtype)
        w_hh = self.weight_hh.value
        for t in reversed(range(t_len)):
            i, f, g, o = (acts[:, t, k * hd:(k + 1) * hd] for k in range(4))
            tc = tcs[:, t]
            dh = grad[:, t] + dh_next
            do = dh * tc
            dc = dc_next + dh * o * (1.0 - tc ** 2)
            di, df, dg = dc * g, dc * cs[:, t], dc * i
            dc_next = dc * f
            dz = np.concatenate([di * i * (1 - i), df * f * (1 - f), dg * (1 - g ** 2), do * o * (1 - o)], axis=1)
            self.weight_hh.grad += dz.T @ hs[:, t]
            dz_all[:, t] = dz
            dh_next = dz @ w_hh
        flat = dz_all.reshape(-1, 4 * hd)
        self.weight_ih.grad += flat.T @ x.reshape(-1, d)
        self.bias_ih.grad += flat.sum(axis=0)
        self.bias_hh.grad += flat.sum(axis=0)
        return dz_all @ self.weight_ih.value

class GRU(_Recurrent):
    """GRU (Cho et al., 2014) theo dạng của PyTorch/cuDNN (Keras ``reset_after=True``).

    r = σ(·), z = σ(·), n = tanh(W_in·x + b_in + r⊙(W_hn·h + b_hn)),  h_t = (1−z)⊙n + z⊙h_{t-1}.
    """

    gates = 3

    def forward(self, x):
        n, t_len, _ = x.shape
        hd = self.hidden_size
        gi = x @ self.weight_ih.value.T + self.bias_ih.value
        h = np.zeros((n, hd), dtype=x.dtype)
        hs = np.empty((n, t_len + 1, hd), dtype=x.dtype); hs[:, 0] = h
        cache = np.empty((n, t_len, 4 * hd), dtype=x.dtype)  # r, z, n, gh_n
        for t in range(t_len):
            gh = h @ self.weight_hh.value.T + self.bias_hh.value
            r = _sigmoid(gi[:, t, :hd] + gh[:, :hd])
            z = _sigmoid(gi[:, t, hd:2 * hd] + gh[:, hd:2 * hd])
            nn_ = np.tanh(gi[:, t, 2 * hd:] + r * gh[:, 2 * hd:])
            h = (1.0 - z) * nn_ + z * h
            hs[:, t + 1] = h
            cache[:, t] = np.concatenate([r, z, nn_, gh[:, 2 * hd:]], axis=1)
        self._cache = (x, hs, cache)
        return hs[:, 1:]

    def backward(self, grad):
        x, hs, cache = self._cache
        n, t_len, d = x.shape
        hd = self.hidden_size
        dgi_all = np.empty((n, t_len, 3 * hd), dtype=grad.dtype)
        dh_next = np.zeros((n, hd), dtype=grad.dtype)
        w_hh = self.weight_hh.value
        for t in reversed(range(t_len)):
            r, z, nn_, ghn = (cache[:, t, k * hd:(k + 1) * hd] for k in range(4))
            h_prev = hs[:, t]
            dh = grad[:, t] + dh_next
            dan = dh * (1.0 - z) * (1.0 - nn_ ** 2)
            daz = dh * (h_prev - nn_) * z * (1.0 - z)
            dar = dan * ghn * r * (1.0 - r)
            dgh = np.concatenate([dar, daz, dan * r], axis=1)
            dgi_all[:, t] = np.concatenate([dar, daz, dan], axis=1)
            self.weight_hh.grad += dgh.T @ h_prev
            self.bias_hh.grad += dgh.sum(axis=0)
            dh_next = dh * z + dgh @ w_hh
        flat = dgi_all.reshape(-1, 3 * hd)
        self.weight_ih.grad += flat.T @ x.reshape(-1, d)
        self.bias_ih.grad += flat.sum(axis=0)
        return dgi_all @ self.weight_ih.value

class LastStep(Module):
    """Lấy trạng thái ẩn ở bước thời gian cuối: [N, T, H] → [N, H]."""

    def forward(self, x):
        self._shape = x.shape
        return x[:, -1]

    def backward(self, grad):
        out = np.zeros(self._shape, dtype=grad.dtype)
        out[:, -1] = grad
        return out

### Mã nền tảng: optim

In [ ]:
"""Thuật toán tối ưu: SGD (có quán tính) và Adam (Kingma & Ba, 2015); cắt chuẩn gradient."""

import numpy as np

class SGD:
    def __init__(self, params: list[Parameter], lr: float = 0.01, momentum: float = 0.0):
        self.params, self.lr, self.momentum = params, lr, momentum
        self.velocity = [np.zeros_like(p.value) for p in params]

    def step(self) -> None:
        for p, v in zip(self.params, self.velocity):
            v *= self.momentum
            v += p.grad
            p.value -= self.lr * v

class Adam:
    """m ← β₁m + (1−β₁)g;  v ← β₂v + (1−β₂)g²;  θ ← θ − η·m̂/(√v̂ + ε)  (m̂, v̂ đã hiệu chỉnh độ lệch).

    ``weight_decay`` > 0 dùng dạng tách rời (decoupled, như AdamW) để thống nhất với Keras 3.
    """

    def __init__(self, params: list[Parameter], lr: float = 1e-3, betas=(0.9, 0.999), eps: float = 1e-8,
                 weight_decay: float = 0.0):
        self.params, self.lr, self.eps, self.weight_decay = params, lr, eps, weight_decay
        self.beta1, self.beta2 = betas
        self.m = [np.zeros_like(p.value) for p in params]
        self.v = [np.zeros_like(p.value) for p in params]
        self.t = 0

    def step(self) -> None:
        self.t += 1
        bc1 = 1.0 - self.beta1 ** self.t
        bc2 = 1.0 - self.beta2 ** self.t
        for p, m, v in zip(self.params, self.m, self.v):
            g = p.grad
            m *= self.beta1
            m += (1.0 - self.beta1) * g
            v *= self.beta2
            v += (1.0 - self.beta2) * g * g
            if self.weight_decay:
                p.value -= self.lr * self.weight_decay * p.value
            p.value -= (self.lr * (m / bc1) / (np.sqrt(v / bc2) + self.eps)).astype(p.value.dtype)

def clip_grad_norm(params: list[Parameter], max_norm: float) -> float:
    """Cắt gradient theo chuẩn toàn cục (Pascanu et al., 2013) — chống bùng nổ gradient trong RNN."""
    total = float(np.sqrt(sum(float(np.sum(p.grad.astype(np.float64) ** 2)) for p in params)))
    coef = max_norm / (total + 1e-6)
    if coef < 1.0:
        for p in params:
            p.grad *= coef
    return total

## 3. Cùng kiến trúc trên NumPy, Keras và PyTorch

In [ ]:
HIDDEN=8
def recurrent_scratch(kind, input_shape, seed):
    rng = np.random.default_rng(seed)
    if kind == "mlp":
        return Sequential(Dense(input_shape[0], 16, rng=rng), ReLU(), Dense(16, 1, rng=rng))
    if kind in ("cnn4", "cnn8"):
        filters = 8 if kind == "cnn8" else 4
        return Sequential(Conv2d(1, filters, 3, rng=rng), ReLU(), MaxPool2d(2),       # 20 → 18 → 9
                          Flatten(), Dense(filters * 9 * 9, 1, rng=rng))
    if kind == "cnndeep":
        return Sequential(Conv2d(1, 4, 3, rng=rng), ReLU(), MaxPool2d(2),             # 20 → 18 → 9
                          Conv2d(4, 8, 3, rng=rng), ReLU(), MaxPool2d(2),             # 9 → 7 → 3
                          Flatten(), Dense(8 * 3 * 3, 1, rng=rng))
    cell = {"rnn": SimpleRNN, "lstm": LSTM, "gru": GRU}[kind]
    return Sequential(cell(input_shape[-1], HIDDEN, rng=rng), LastStep(), Dense(HIDDEN, 1, rng=rng))

def recurrent_torch(kind, input_shape):
    import torch
    from torch import nn

    torch.set_num_threads(2)
    if kind == "mlp":
        return nn.Sequential(nn.Linear(input_shape[0], 16), nn.ReLU(), nn.Linear(16, 1))
    if kind in ("cnn4", "cnn8"):
        filters = 8 if kind == "cnn8" else 4
        return nn.Sequential(nn.Conv2d(1, filters, 3), nn.ReLU(), nn.MaxPool2d(2),
                             nn.Flatten(), nn.Linear(filters * 9 * 9, 1))
    if kind == "cnndeep":
        return nn.Sequential(nn.Conv2d(1, 4, 3), nn.ReLU(), nn.MaxPool2d(2),
                             nn.Conv2d(4, 8, 3), nn.ReLU(), nn.MaxPool2d(2),
                             nn.Flatten(), nn.Linear(8 * 3 * 3, 1))

    class RecurrentNet(nn.Module):
        def __init__(self):
            super().__init__()
            cell = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[kind]
            self.recurrent = cell(input_shape[-1], HIDDEN, batch_first=True)
            self.output = nn.Linear(HIDDEN, 1)

        def forward(self, x):                      # x: [batch, 20, 1]
            sequence, _ = self.recurrent(x)
            return self.output(sequence[:, -1])     # trạng thái ở bước cuối → logit

    return RecurrentNet()

def recurrent_keras(kind, input_shape):
    import keras
    import tensorflow as tf

    try:
        tf.config.threading.set_intra_op_parallelism_threads(2)
        tf.config.threading.set_inter_op_parallelism_threads(2)
    except RuntimeError:  # đã khởi tạo trước đó trong cùng tiến trình
        pass
    inputs = keras.Input(shape=input_shape)
    z = inputs
    if kind == "mlp":
        z = keras.layers.Dense(16, activation="relu")(z)
    elif kind.startswith("cnn"):
        z = keras.layers.Permute((2, 3, 1))(z)      # [C, H, W] → [H, W, C] (quy ước của Keras)
        first = 8 if kind == "cnn8" else 4
        z = keras.layers.Conv2D(first, 3, activation="relu")(z)
        z = keras.layers.MaxPooling2D(2)(z)
        if kind == "cnndeep":
            z = keras.layers.Conv2D(8, 3, activation="relu")(z)
            z = keras.layers.MaxPooling2D(2)(z)
        z = keras.layers.Permute((3, 1, 2))(z)      # trả về [C, H, W] để Flatten cùng thứ tự với PyTorch
        z = keras.layers.Flatten()(z)
    else:
        cell = {"rnn": keras.layers.SimpleRNN, "lstm": keras.layers.LSTM, "gru": keras.layers.GRU}[kind]
        z = cell(HIDDEN)(z)                          # mặc định trả về trạng thái ở bước cuối
    outputs = keras.layers.Dense(1)(z)               # logit (chưa qua sigmoid)
    return keras.Model(inputs, outputs)

def _scratch_layers_with_weights(model):
    return [layer for layer in model.layers if layer.parameters()]

def copy_weights_to_torch(scratch, model, kind):
    import torch

    with torch.no_grad():
        for target, source in zip(model.parameters(), scratch.parameters(), strict=True):
            target.copy_(torch.from_numpy(source.value))
    if kind in ("rnn", "lstm"):
        # PyTorch có hai bias cộng dồn (b_ih + b_hh), Keras chỉ có một. Giữ b_hh = 0 và không học
        # để số tham số học và phép toán tương đương ở cả ba bản cài đặt.
        model.recurrent.bias_hh_l0.requires_grad_(False)

def copy_weights_to_keras(scratch, model, kind):
    targets = [layer for layer in model.layers if layer.weights]
    for target, source in zip(targets, _scratch_layers_with_weights(scratch), strict=True):
        if isinstance(source, Dense):
            target.set_weights([source.weight.value.T, source.bias.value])
        elif isinstance(source, Conv2d):          # [F, C, KH, KW] → [KH, KW, C, F]
            target.set_weights([source.weight.value.transpose(2, 3, 1, 0), source.bias.value])
        elif kind in ("rnn", "lstm"):              # thứ tự cổng LSTM (i, f, g, o) trùng nhau
            target.set_weights([source.weight_ih.value.T, source.weight_hh.value.T,
                                source.bias_ih.value + source.bias_hh.value])
        else:                                      # GRU: Keras xếp cổng (z, r, n), PyTorch/NumPy (r, z, n)
            h = HIDDEN
            order = np.r_[h:2 * h, 0:h, 2 * h:3 * h]
            target.set_weights([source.weight_ih.value[order].T, source.weight_hh.value[order].T,
                                np.stack([source.bias_ih.value[order], source.bias_hh.value[order]])])

In [ ]:
"""Ba cách cài đặt dùng chung kiến trúc, khởi tạo và hàm mục tiêu.

Chỉ nhập thư viện học sâu khi được yêu cầu; suy luận NumPy dùng được trên Render.
"""

import os

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

os.environ.setdefault('OMP_NUM_THREADS', '2')

import numpy as np

KINDS = {'tabular': ('logistic','svm','mlp'),
         'image': ('cnn4','cnn8','cnndeep'), 'sequence': ('rnn','lstm','gru')}

def scratch_model(kind, shape, seed=11, n_classes=10):
    rng = np.random.default_rng(seed)
    if kind in ('rnn','lstm','gru'):
        return recurrent_scratch(kind, shape, seed)
    if kind in ('logistic','svm'):
        return Sequential(Dense(shape[0], 1, rng=rng))
    if kind == 'mlp':
        return Sequential(Dense(shape[0],16,rng=rng), ReLU(), Dense(16,1,rng=rng))
    filters = 8 if kind == 'cnn8' else 4
    layers = [Conv2d(1,filters,3,rng=rng), ReLU(), MaxPool2d(2)]
    side = (shape[-1]-2)//2
    if kind == 'cnndeep':
        layers += [Conv2d(filters,8,3,rng=rng), ReLU(), MaxPool2d(2)]
        filters, side = 8, (side-2)//2
    return Sequential(*layers, Flatten(), Dense(filters*side*side,n_classes,rng=rng))

def torch_model(kind, shape, n_classes=10):
    import torch
    from torch import nn
    torch.set_num_threads(2)
    if kind in ('rnn','lstm','gru'):
        return recurrent_torch(kind,shape)
    if kind in ('logistic','svm'):
        return nn.Sequential(nn.Linear(shape[0],1))
    if kind == 'mlp':
        return nn.Sequential(nn.Linear(shape[0],16), nn.ReLU(), nn.Linear(16,1))
    filters = 8 if kind == 'cnn8' else 4
    layers = [nn.Conv2d(1,filters,3), nn.ReLU(), nn.MaxPool2d(2)]
    side = (shape[-1]-2)//2
    if kind == 'cnndeep':
        layers += [nn.Conv2d(filters,8,3), nn.ReLU(), nn.MaxPool2d(2)]
        filters, side = 8, (side-2)//2
    return nn.Sequential(*layers,nn.Flatten(),nn.Linear(filters*side*side,n_classes))

def keras_model(kind, shape, n_classes=10):
    import keras
    import tensorflow as tf
    try:
        tf.config.threading.set_intra_op_parallelism_threads(2)
        tf.config.threading.set_inter_op_parallelism_threads(2)
    except RuntimeError:
        pass
    if kind in ('rnn','lstm','gru'):
        return recurrent_keras(kind,shape)
    x = keras.Input(shape=shape)
    z = x
    if kind == 'mlp':
        z = keras.layers.Dense(16,activation='relu')(z)
    if kind.startswith('cnn'):
        z = keras.layers.Permute((2,3,1))(z)
        z = keras.layers.Conv2D(8 if kind=='cnn8' else 4,3,activation='relu')(z)
        z = keras.layers.MaxPooling2D(2)(z)
        if kind == 'cnndeep':
            z = keras.layers.Conv2D(8,3,activation='relu')(z)
            z = keras.layers.MaxPooling2D(2)(z)
        z = keras.layers.Permute((3,1,2))(z)
        z = keras.layers.Flatten()(z)
    return keras.Model(x,keras.layers.Dense(n_classes if kind.startswith('cnn') else 1)(z))

def build_models(kind, shape, seed=11, frameworks=('scratch','keras','pytorch'), n_classes=10):
    source = scratch_model(kind,shape,seed,n_classes)
    models = {'scratch':source}
    if 'pytorch' in frameworks:
        models['pytorch'] = torch_model(kind,shape,n_classes)
        copy_weights_to_torch(source,models['pytorch'],kind)
    if 'keras' in frameworks:
        models['keras'] = keras_model(kind,shape,n_classes)
        copy_weights_to_keras(source,models['keras'],kind)
    return {fw:models[fw] for fw in frameworks}

def loss_and_gradient(z, y, kind):
    """Loss dữ liệu; SVM thêm phạt L2 cho trọng số ở vòng cập nhật."""
    z = np.asarray(z,dtype=np.float64)
    y = np.asarray(y,dtype=int).reshape(-1)
    if kind.startswith('cnn'):
        shifted = z-z.max(axis=1,keepdims=True)
        exp = np.exp(shifted)
        probability = exp/exp.sum(axis=1,keepdims=True)
        loss = np.mean(np.log(exp.sum(axis=1))-shifted[np.arange(len(y)),y])
        probability[np.arange(len(y)),y] -= 1
        return float(loss), probability/len(y)
    signed = (2*y-1)[:,None]
    if kind == 'svm':
        margin = 1-signed*z
        return float(np.maximum(0,margin).mean()), -signed*(margin>0)/len(y)
    target = y[:,None]
    probability = .5*(1+np.tanh(z/2))
    return float(np.mean(np.logaddexp(0,z)-target*z)), (probability-target)/len(y)

def logits(model, framework, x, batch_size=128):
    output=[]
    if framework in ('scratch','pytorch'):
        model.eval()
    for offset in range(0,len(x),batch_size):
        batch=x[offset:offset+batch_size]
        if framework=='scratch':
            value=model.forward(batch)
        elif framework=='keras':
            value=model(batch,training=False).numpy()
        else:
            import torch
            with torch.no_grad():
                value=model(torch.from_numpy(batch)).numpy()
        output.append(value)
    return np.concatenate(output)

def probabilities(z):
    if z.shape[1]==1:
        p=.5*(1+np.tanh(z[:,0]/2))
        return np.column_stack([1-p,p])
    exp=np.exp(z-z.max(axis=1,keepdims=True))
    return exp/exp.sum(axis=1,keepdims=True)

## 4. Kiểm tra trước khi huấn luyện

    So sánh logits trên cùng đầu vào và cùng trọng số. Sai số nhỏ do phép toán số thực;
    nếu kiểm tra thất bại phải sửa trước khi chạy dài. Bước này không huấn luyện toàn bộ dữ liệu.

In [ ]:
for key in DATASETS:
    data,meta=load_csv_dataset(ROOT,key)
    sample=batch_input(data['train_x'][:4])
    for kind in MODEL_KINDS:
        models=build_models(kind,tuple(meta['input_shape']),CONFIG['seed'],n_classes=len(meta['classes']))
        reference=logits(models['scratch'],'scratch',sample)
        for fw in ('pytorch','keras'):
            actual=logits(models[fw],fw,sample)
            error=float(np.max(np.abs(reference-actual)))
            np.testing.assert_allclose(reference,actual,atol=2e-5,rtol=1e-4)
            print(key,kind,fw,'– sai số logits:',error)
        del models
    del data

## 5. Vòng học, checkpoint và đánh giá

    Cùng thứ tự mini-batch, Adam và clipping. Chọn epoch bằng validation loss, không bằng test.
    SVM dùng hinge loss và phạt L2, điểm SVM không được gọi là xác suất.
    Một seed là đối chiếu triển khai, chưa đủ kết luận thống kê về thứ hạng các mô hình.
    Khi đổi mã hoặc cấu hình hãy đổi RUN_ID để không lẫn kết quả.

In [ ]:
"""Vòng huấn luyện hiện đầy đủ trong notebook; lưu checkpoint sau mỗi epoch.

Checkpoint pickle chỉ được đọc từ tệp cục bộ do chính notebook tạo.
Không tải hoặc mở checkpoint pickle từ người khác.
"""

import copy

import hashlib

import json

from pathlib import Path

import pickle

import time

import numpy as np

from sklearn.metrics import accuracy_score,balanced_accuracy_score,f1_score,roc_auc_score,confusion_matrix

class BatchTrainer:
    def __init__(self,model,framework,kind,learning_rate=.001):
        self.model,self.framework,self.kind=model,framework,kind
        if framework=='scratch':
            self.params=[p for p in model.parameters() if not(kind in ('rnn','lstm') and p.name=='bias_hh')]
            self.optimizer=Adam(self.params,lr=learning_rate,eps=1e-8)
        elif framework=='pytorch':
            import torch
            self.params=[p for p in model.parameters() if p.requires_grad]
            self.optimizer=torch.optim.Adam(self.params,lr=learning_rate,eps=1e-8)
        else:
            import keras
            import tensorflow as tf
            self.optimizer=keras.optimizers.Adam(learning_rate=learning_rate,epsilon=1e-8,global_clipnorm=1.)
            self.optimizer.build(model.trainable_variables)
            self.graph=tf.function(self.keras_step,reduce_retracing=True)

    def keras_step(self,x,y):
        import tensorflow as tf
        with tf.GradientTape() as tape:
            z=self.model(x,training=True)
            if self.kind.startswith('cnn'):
                loss=tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y,logits=z))
            elif self.kind=='svm':
                target=2*tf.cast(y[:,None],tf.float32)-1
                loss=tf.reduce_mean(tf.nn.relu(1-target*z))+.0005*tf.reduce_sum(self.model.trainable_variables[0]**2)
            else:
                loss=tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=tf.cast(y[:,None],tf.float32),logits=z))
        gradients=tape.gradient(loss,self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients,self.model.trainable_variables))
        return loss

    def step(self,x,y):
        if self.framework=='keras':
            return float(self.graph(x,y).numpy())
        self.model.train()
        if self.framework=='scratch':
            self.model.zero_grad()
            z=self.model.forward(x)
            loss,gradient=loss_and_gradient(z,y,self.kind)
            self.model.backward(gradient.astype('float32'))
            if self.kind=='svm':
                loss+=.0005*float(np.sum(self.params[0].value**2))
                self.params[0].grad+=.001*self.params[0].value
            clip_grad_norm(self.params,1.)
            self.optimizer.step()
            return loss
        import torch
        self.optimizer.zero_grad()
        z=self.model(torch.from_numpy(x)); target=torch.from_numpy(y)
        if self.kind.startswith('cnn'):
            loss=torch.nn.functional.cross_entropy(z,target.long())
        elif self.kind=='svm':
            loss=torch.relu(1-(2*target[:,None]-1)*z).mean()+.0005*(self.params[0]**2).sum()
        else:
            loss=torch.nn.functional.binary_cross_entropy_with_logits(z,target[:,None].float())
        loss.backward(); torch.nn.utils.clip_grad_norm_(self.params,1.)
        self.optimizer.step()
        return float(loss.detach())

    def state(self):
        if self.framework=='scratch':
            return copy.deepcopy({'m':self.optimizer.m,'v':self.optimizer.v,'t':self.optimizer.t})
        if self.framework=='pytorch':
            return copy.deepcopy(self.optimizer.state_dict())
        return [v.numpy() for v in self.optimizer.variables]

    def restore(self,state):
        if self.framework=='scratch':
            self.optimizer.m,self.optimizer.v,self.optimizer.t=copy.deepcopy(state['m']),copy.deepcopy(state['v']),state['t']
        elif self.framework=='pytorch':
            self.optimizer.load_state_dict(state)
        else:
            for variable,value in zip(self.optimizer.variables,state,strict=True):
                variable.assign(value)

def model_state(model,framework):
    return model.get_weights() if framework=='keras' else copy.deepcopy(model.state_dict())

def restore_model(model,framework,state):
    model.set_weights(state) if framework=='keras' else model.load_state_dict(state)

def infer_batches(model,framework,x,batch_size=256):
    outputs=[]
    if framework!='keras':
        model.eval()
    for start in range(0,len(x),batch_size):
        xb=batch_input(x[start:start+batch_size])
        if framework=='scratch':
            value=model.forward(xb)
        elif framework=='keras':
            value=model(xb,training=False).numpy()
        else:
            import torch
            with torch.no_grad():
                value=model(torch.from_numpy(xb)).numpy()
        outputs.append(value)
    return np.concatenate(outputs)

def fit_large(model,framework,kind,data,config,checkpoint,signature):
    """Khôi phục cả trọng số, Adam, RNG, patience, checkpoint tốt nhất và lịch sử."""
    trainer=BatchTrainer(model,framework,kind,config['learning_rate'])
    rng=np.random.default_rng(config['seed']+1000)
    state={'signature':signature,'history':[],'best_loss':float('inf'),'best_model':None,
           'best_epoch':0,'wait':0,'training_seconds':0.}
    checkpoint=Path(checkpoint)
    if checkpoint.exists():
        with checkpoint.open('rb') as handle:
            saved=pickle.load(handle)
        if saved['signature']!=signature:
            raise ValueError('Checkpoint khác dữ liệu/mã/cấu hình. Chọn RUN_ID mới; không ghi đè lặng lẽ.')
        state=saved
        restore_model(model,framework,state['model'])
        trainer.restore(state['optimizer'])
        rng.bit_generator.state=state['rng']
        print('Tiếp tục sau epoch',len(state['history']))
    for epoch in range(len(state['history']),config['epochs']):
        if state['wait']>=config['patience']:
            break
        started=time.perf_counter(); total=0.
        order=rng.permutation(len(data['train_y']))
        for offset in range(0,len(order),config['batch_size']):
            ids=order[offset:offset+config['batch_size']]
            total+=trainer.step(batch_input(data['train_x'][ids]),np.asarray(data['train_y'][ids],dtype='int64'))*len(ids)
        val=loss_and_gradient(infer_batches(model,framework,data['val_x']),data['val_y'],kind)[0]
        if not np.isfinite(val):
            raise ValueError('Validation loss không hữu hạn; không lưu kết quả sai.')
        elapsed=time.perf_counter()-started
        state['history'].append({'epoch':epoch+1,'train_loss':total/len(order),'val_loss':val,'seconds':elapsed})
        state['training_seconds']+=elapsed
        if val<state['best_loss']-1e-6:
            state.update(best_loss=val,best_model=model_state(model,framework),best_epoch=epoch+1,wait=0)
        else:
            state['wait']+=1
        state.update(model=model_state(model,framework),optimizer=trainer.state(),rng=rng.bit_generator.state)
        checkpoint.parent.mkdir(parents=True,exist_ok=True)
        temporary=checkpoint.with_suffix('.tmp')
        with temporary.open('wb') as handle:
            pickle.dump(state,handle,protocol=pickle.HIGHEST_PROTOCOL)
        temporary.replace(checkpoint)
        print(f"Epoch {epoch+1}/{config['epochs']} | học {total/len(order):.4f} | validation {val:.4f} | {elapsed:.1f} giây",flush=True)
    restore_model(model,framework,state['best_model'])
    return {k:state[k] for k in ('history','best_epoch','best_loss','training_seconds')}

def evaluate_large(y,z):
    pred=z.argmax(1) if z.shape[1]>1 else (z[:,0]>=0).astype(int)
    result={'accuracy':accuracy_score(y,pred),'balanced_accuracy':balanced_accuracy_score(y,pred),
        'macro_f1':f1_score(y,pred,average='macro',zero_division=0),
        'confusion_matrix':confusion_matrix(y,pred).tolist()}
    # ROC-AUC đa lớp chỉ có nghĩa khi tất cả các lớp đều có trong tập test.
    if z.shape[1]==1:
        result['roc_auc']=roc_auc_score(y,z[:,0])
    elif len(np.unique(y))==z.shape[1]:
        result['roc_auc']=roc_auc_score(y,probabilities(z),multi_class='ovr',average='macro')
    return result,pred

def run_experiment(root,key,kind,framework,data,meta,config,run_id,code_hash):
    enforce_size(len(data['train_y']))
    folder=Path(root)/'results/tieuluan/course_large'/run_id
    folder.mkdir(parents=True,exist_ok=True)
    stem=f'{key}__{kind}__{framework}__{config["seed"]}'
    signature=hashlib.sha256(json.dumps({'arrays':meta['array_hashes'],'config':config,'code':code_hash,
        'key':key,'kind':kind,'framework':framework},sort_keys=True).encode()).hexdigest()
    record_path=folder/f'{stem}.json'
    if record_path.exists():
        record=json.loads(record_path.read_text(encoding='utf8'))
        if record['signature']!=signature:
            raise ValueError('Kết quả cũ khác cấu hình. Đổi RUN_ID để tách thực nghiệm.')
        if not (folder/f'{stem}_predictions.npz').exists():
            raise ValueError('Thiếu tệp dự báo của lượt chạy đã hoàn thành.')
        print('Đã hoàn thành, bỏ qua:',stem)
        return record
    print('Bắt đầu:',stem,flush=True)
    model=build_models(kind,tuple(meta['input_shape']),config['seed'],frameworks=(framework,),n_classes=len(meta['classes']))[framework]
    history=fit_large(model,framework,kind,data,config,folder/'checkpoints'/f'{stem}.pkl',signature)
    z=infer_batches(model,framework,data['test_x'])
    metrics,pred=evaluate_large(data['test_y'],z)
    baselines={}
    majority=int(np.bincount(data['train_y']).argmax())
    for name,baseline in [('majority',np.full(len(pred),majority)),('persistence',data.get('test_persistence'))]:
        if baseline is not None:
            baselines[name]={'accuracy':accuracy_score(data['test_y'],baseline),
                'balanced_accuracy':balanced_accuracy_score(data['test_y'],baseline)}
    model_dir=Path(root)/'models/tieuluan/course_large'/run_id
    model_dir.mkdir(parents=True,exist_ok=True)
    if framework=='scratch':
        model.save(model_dir/f'{stem}.npz')
    elif framework=='keras':
        model.save_weights(model_dir/f'{stem}.weights.h5')
    else:
        import torch
        torch.save(model.state_dict(),model_dir/f'{stem}.pt')
    np.savez_compressed(folder/f'{stem}_predictions.npz',y=data['test_y'],logits=z,prediction=pred,ids=data['test_ids'])
    record={'dataset':key,'kind':kind,'framework':framework,'config':config,'signature':signature,
        'code_hash':code_hash,'metadata':meta,'metrics':metrics,'baselines':baselines,**history}
    temporary=record_path.with_suffix('.tmp')
    temporary.write_text(json.dumps(record,ensure_ascii=False,indent=2),encoding='utf8')
    temporary.replace(record_path)
    return record

## 6. Huấn luyện đầy đủ – ô chạy lâu

    Chạy ô này khi máy đã sẵn sàng. Có thể thu hẹp MODEL_KINDS/FRAMEWORKS để chạy từng lượt,
    nhưng phải hoàn thành tất cả tổ hợp trước khi tổng hợp báo cáo. Lượt hoàn thành được bỏ qua;
    lượt bị ngắt tiếp tục từ epoch đã lưu. Số mẫu không bị cắt giảm theo chế độ chạy.

In [ ]:
records=[]
for key in DATASETS:
    data,meta=load_csv_dataset(ROOT,key)
    for kind in MODEL_KINDS:
        for framework in FRAMEWORKS:
            result=run_experiment(ROOT,key,kind,framework,data,meta,CONFIG,RUN_ID,CODE_HASH)
            records.append(result)
    del data
display(pd.DataFrame([{'Dữ liệu':r['dataset'],'Mô hình':r['kind'],'Cài đặt':r['framework'],
    'Đúng cân bằng':r['metrics']['balanced_accuracy'],'F1 trung bình':r['metrics']['macro_f1'],
    'Giây huấn luyện':r['training_seconds']} for r in records]))

## 7. Hình, bảng và mã đưa vào báo cáo

    Chỉ chạy sau khi đủ kết quả. Thiếu lượt nào thì công cụ báo thiếu, không điền số giả.
    Đọc các mẫu đoán sai, đường học và so sánh với cách đoán đơn giản trước khi viết nhận xét.

In [ ]:
from src.tieuluan.course.large_reporting import export_chapter
report_path=export_chapter(ROOT,RUN_ID,DATASETS,MODEL_KINDS,CONFIG['seed'],NOTEBOOK_PATH)
print('Đã xuất nội dung và tài nguyên báo cáo:',report_path)
for key in DATASETS:
    for kind in MODEL_KINDS:
        fig,ax=plt.subplots(figsize=(7,3))
        for fw in ('scratch','pytorch','keras'):
            path=ROOT/'results/tieuluan/course_large'/RUN_ID/f"{key}__{kind}__{fw}__{CONFIG['seed']}.json"
            item=json.loads(path.read_text(encoding='utf8'))
            ax.plot([p['epoch'] for p in item['history']],[p['val_loss'] for p in item['history']],label=fw)
        ax.set(title=key+' / '+kind,xlabel='Vòng học',ylabel='Mất mát validation'); ax.legend(); plt.show()